# Setup

In [1]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras

In [2]:
print("tf:", tf.__version__)
print("keras:", keras.__version__)

tf: 2.12.0
keras: 2.12.0


In [ ]:
def permutation_importance_global_sequence(
    model,
    X_val,                  # shape: (N, T, F)
    y_val,                  # shape: (N, T) or model-compatible target
    feature_names=None,
    n_repeats=5,
    batch_size=128,
    sample_weight=None,     # optional mask/weights, shape usually (N, T)
    shuffle_mode="sequence",# "sequence" or "per_timestep"
    random_state=42,
):
    """
    Global permutation importance for sequence models (GRU/LSTM), no retraining.
    Importance = mean(permuted_loss - baseline_loss) across repeats.
    """

    rng = np.random.default_rng(random_state)

    # 1) Baseline validation loss (trained model, no retraining)
    baseline = model.evaluate(
        X_val, y_val,
        sample_weight=sample_weight,
        batch_size=batch_size,
        verbose=0,
        return_dict=True
    )
    baseline_loss = baseline["loss"]

    n_features = X_val.shape[2]
    if feature_names is None:
        feature_names = [f"feature_{i}" for i in range(n_features)]

    results = []

    # 2) Permute one feature at a time, recompute loss
    for j in range(n_features):
        deltas = []

        for _ in range(n_repeats):
            Xp = X_val.copy()

            if shuffle_mode == "sequence":
                # Shuffle entire trajectory of feature j across samples:
                # preserves within-trial temporal structure for that feature.
                perm_idx = rng.permutation(X_val.shape[0])
                Xp[:, :, j] = X_val[perm_idx, :, j]

            elif shuffle_mode == "per_timestep":
                # Shuffle feature j across samples independently at each time t.
                for t in range(X_val.shape[1]):
                    perm_idx = rng.permutation(X_val.shape[0])
                    Xp[:, t, j] = X_val[perm_idx, t, j]
            else:
                raise ValueError("shuffle_mode must be 'sequence' or 'per_timestep'")

            perm_eval = model.evaluate(
                Xp, y_val,
                sample_weight=sample_weight,
                batch_size=batch_size,
                verbose=0,
                return_dict=True
            )
            perm_loss = perm_eval["loss"]
            deltas.append(perm_loss - baseline_loss)  # degradation

        results.append({
            "feature": feature_names[j],
            "importance_mean": float(np.mean(deltas)),
            "importance_std": float(np.std(deltas)),
            "baseline_loss": float(baseline_loss),
            "permuted_loss_mean": float(baseline_loss + np.mean(deltas)),
        })

    ranking = pd.DataFrame(results).sort_values("importance_mean", ascending=False).reset_index(drop=True)
    return ranking

In [5]:
model_name = "rlf"
N_AGENTS = 500
num_blocks = 12
max_num_trials = 78
suffix = '3ParamRL_no_switch_st22'
filename = f"./data/{model_name}_6s3a/{N_AGENTS}a_{num_blocks}b_{max_num_trials}t_{suffix}.csv"
print(filename)
test_data = pd.read_csv(filename)

./data/rlf_6s3a/500a_12b_78t_3ParamRL_no_switch_st22.csv


In [6]:
from utils.feature_utils import (
    get_iter_acc_without_switches,
    get_iter_acc_with_switches,
    extract_features_blockless,
)

%matplotlib inline
import warnings; warnings.simplefilter('ignore')  # hide warnings
feature_list = ['actions', 'rewards',  'stimuli','isswitch','delay_since_last_stimuli']

has_switches = (test_data.isswitch.nunique() == 2)
test_iter = get_iter_acc_with_switches(test_data) if has_switches else get_iter_acc_without_switches(test_data)
test_iter['isswitch'] = test_iter['trials'].apply(lambda x: 1 if x == 0 else 0)
test_iter['prev_action'] = test_data.groupby(['agentid', 'block_no'])['actions'].shift(1).fillna(-1).astype(int).to_list()
test_iter['delay_since_last_stimuli'] = test_data['delay_since_last_stimuli'].to_list()

test_features = extract_features_blockless(test_iter, feature_list, [])
#test_features = _extract_features_blockless(test_iter, feature_list, False)#get_block_onehot_features(short_test_data, feature_list) #

num_test_agents = test_data['agentid'].nunique()
n_trial = test_data['trials'].nunique()
n_block = test_data['block_no'].nunique()
qv = np.array(test_data['rewards'] - test_data['rpe_history'])
test_labels = qv.astype(np.float32).reshape((num_test_agents, n_block*n_trial))

print(test_features.shape, test_labels.shape)

(500, 936, 5) (500, 936)


In [ ]:
model_name = "lasenet_RLF-6S3A_a-r-s-i-d_A3000_936t_B128_U256_D0.097_D0.013_D0.011_0.000696_model.keras"
gru_model = tf.keras.models.load_model(f"./models/{model_name}", compile=False)
# gru_model = tf.keras.models.load_model(f"./models/{model_name}_6s3a_seq1_{N_AGENTS}a_{num_blocks}b_{max_num_trials}t_{suffix}.h5")

TypeError: Could not deserialize class 'Functional' because its parent module keras.src.models.functional cannot be imported. Full object config: {'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {'name': 'functional', 'trainable': True, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, None, 5], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer', 'optional': False}, 'registered_name': None, 'name': 'input_layer', 'inbound_nodes': []}, {'module': 'keras.layers', 'class_name': 'Bidirectional', 'config': {'name': 'bidirectional', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'merge_mode': 'concat', 'layer': {'module': 'keras.layers', 'class_name': 'GRU', 'config': {'name': 'forward_gru', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'regi

In [ ]:
# X_test: (500, 936, 5), y_test: (500, 936)
feature_names = feature_list

ranking = permutation_importance_global_sequence(
    model=gru_model,     # already trained once
    X_val=test_features,
    y_val=test_labels,
    feature_names=feature_names,
    n_repeats=10,
    batch_size=64,
    sample_weight=None,  # put mask here if you use padded timesteps
    shuffle_mode="sequence",
    random_state=123
)

print(ranking)